# NetCDF auf Zell-/Grid-Geometrie reduzieren

Dieses Notebook erstellt aus einer NetCDF-Datei eine kleinere Datei, die nur noch die Grid-Geometrie enthält.

Behalten werden standardmässig:

- `lat` und `lon` als unveränderte 2D-Arrays
- `x` und `y`, falls vorhanden

Damit funktioniert dein späteres KD-Tree-Indexing weiterhin mit `ravel()` und `np.unravel_index(...)`.


## 1. Benötigte Pakete

Falls `xarray`, `netCDF4` oder `scipy` noch fehlen, kannst du sie mit der folgenden Zelle installieren.


In [ ]:
# Optional ausführen, falls Pakete fehlen
# %pip install xarray netCDF4 scipy numpy

## 2. Imports und Pfade

Passe `nc_filepath` und `out_filepath` bei Bedarf an.


In [1]:
from pathlib import Path

import numpy as np
import xarray as xr
from scipy.spatial import cKDTree

# Eingabedatei
nc_filepath = Path("nc_folder/NC_for_Cellid_alt.nc")

# Ausgabedatei mit nur noch Grid-Geometrie
out_filepath = Path("nc_folder/NC_for_Cellid.nc")

# Variablennamen in der NetCDF-Datei
lat_name = "lat"
lon_name = "lon"

# x/y-Koordinaten behalten, falls vorhanden
keep_xy = True

# Kompression der Output-Datei: 0 = keine, 9 = maximal
compression_level = 4

# False empfohlen, damit lat/lon numerisch exakt wie im Original bleiben.
# True spart zusätzlich Speicherplatz, erzeugt aber minimale Rundungsdifferenzen.
cast_float32 = False

## 3. Originaldatei kurz prüfen

Diese Zelle zeigt dir Dimensionen, Koordinaten und Variablen der Originaldatei.


In [2]:
ds = xr.open_dataset(nc_filepath, decode_times=False)
ds

<xarray.Dataset> Size: 381MB
Dimensions:      (eps: 10, ref_time: 1, lead_time: 34, y: 295, x: 429)
Coordinates:
  * eps          (eps) int64 80B 1 2 3 4 5 6 7 8 9 10
  * ref_time     (ref_time) int64 8B 0
  * lead_time    (lead_time) int64 272B 0 1 2 3 4 5 6 7 ... 27 28 29 30 31 32 33
    valid_time   (ref_time, lead_time) int64 272B ...
    lon          (y, x) float64 1MB ...
    lat          (y, x) float64 1MB ...
Dimensions without coordinates: y, x
Data variables:
    TOT_PREC     (eps, ref_time, lead_time, y, x) float64 344MB ...
    hourly_rain  (lead_time, y, x) float64 34MB ...

## 4. Funktion: NetCDF auf Geometrie reduzieren

Wichtig: `lat` und `lon` werden nicht geflattet und nicht sortiert. Shape und Reihenfolge bleiben unverändert.


In [3]:
def reduce_to_geometry(
    nc_in: str | Path,
    nc_out: str | Path,
    lat_name: str = "lat",
    lon_name: str = "lon",
    keep_xy: bool = True,
    compression_level: int = 4,
    cast_float32: bool = False,
) -> Path:
    """Reduziert eine NetCDF-Datei auf die Grid-/Zell-Geometrie.

    Behalten werden lat/lon als unveränderte 2D-Arrays sowie optional x/y.
    """
    nc_in = Path(nc_in)
    nc_out = Path(nc_out)

    if not nc_in.exists():
        raise FileNotFoundError(f"Input-Datei nicht gefunden: {nc_in}")

    ds_in = xr.open_dataset(nc_in, decode_times=False)

    missing = [name for name in (lat_name, lon_name) if name not in ds_in.variables]
    if missing:
        raise KeyError(
            f"Diese Variable(n) wurden nicht gefunden: {missing}. "
            f"Vorhanden sind: {list(ds_in.variables)}"
        )

    # Nur lat/lon behalten. x/y bleiben oft automatisch als Koordinaten erhalten,
    # wenn lat/lon diese Dimensionen verwenden.
    out = ds_in[[lat_name, lon_name]].copy()

    # Optional x/y explizit behalten, falls sie vorhanden sind.
    if keep_xy:
        for coord_name in ("x", "y"):
            if coord_name in ds_in.variables and coord_name not in out.variables:
                out = out.assign_coords({coord_name: ds_in[coord_name]})

    # Sicherheitscheck: Für dein KD-Tree-Indexing müssen Form und Reihenfolge gleich bleiben.
    if out[lat_name].shape != ds_in[lat_name].shape:
        raise RuntimeError("lat-Shape wurde verändert.")
    if out[lon_name].shape != ds_in[lon_name].shape:
        raise RuntimeError("lon-Shape wurde verändert.")

    if cast_float32:
        out[lat_name] = out[lat_name].astype("float32")
        out[lon_name] = out[lon_name].astype("float32")

    # Kompression nur für Datenvariablen, nicht für reine Koordinaten.
    encoding = {}
    for name in out.data_vars:
        encoding[name] = {
            "zlib": True,
            "complevel": compression_level,
            "shuffle": True,
        }
        if cast_float32:
            encoding[name]["dtype"] = "float32"

    # Alte Encoding-Metadaten entfernen, damit keine Referenzen auf gelöschte Variablen bleiben.
    for name in out.variables:
        out[name].encoding = {}

    nc_out.parent.mkdir(parents=True, exist_ok=True)
    out.to_netcdf(nc_out, engine="netcdf4", encoding=encoding)

    ds_in.close()
    out.close()

    return nc_out

## 5. Reduzierte Datei schreiben

In [4]:
geometry_file = reduce_to_geometry(
    nc_filepath,
    out_filepath,
    lat_name=lat_name,
    lon_name=lon_name,
    keep_xy=keep_xy,
    compression_level=compression_level,
    cast_float32=cast_float32,
)

print(f"Fertig: {geometry_file}")
print(f"Originalgrösse: {nc_filepath.stat().st_size / 1024**2:.2f} MB")
print(f"Neue Grösse:     {geometry_file.stat().st_size / 1024**2:.2f} MB")

Fertig: nc_folder\NC_for_Cellid.nc
Originalgrösse: 363.06 MB
Neue Grösse:     1.94 MB


## 6. Neue Datei kontrollieren

Hier solltest du nur noch `lat`, `lon` und ggf. `x`, `y` sehen.


In [5]:
ds_geo = xr.open_dataset(out_filepath, decode_times=False)
ds_geo

<xarray.Dataset> Size: 2MB
Dimensions:  (y: 295, x: 429)
Coordinates:
    lat      (y, x) float64 1MB ...
    lon      (y, x) float64 1MB ...
Dimensions without coordinates: y, x
Data variables:
    *empty*

## 7. Test: KD-Tree und Grid-Indizes

Diese Zelle entspricht deinem späteren Zugriff. Sie prüft, ob `lat` und `lon` weiterhin korrekt als 2D-Grid verwendbar sind.


In [6]:
lat = ds_geo[lat_name].values
lon = ds_geo[lon_name].values

print("lat shape:", lat.shape)
print("lon shape:", lon.shape)

# KD-Tree
points = np.column_stack([lon.ravel(), lat.ravel()])
tree = cKDTree(points)

# Grid-Indizes
flat_idx = np.arange(len(points))
i_all, j_all = np.unravel_index(flat_idx, lat.shape)

print("points shape:", points.shape)
print("i_all shape:", i_all.shape)
print("j_all shape:", j_all.shape)
print("Beispiel Punkt 0:", points[0], "-> Grid-Index:", (i_all[0], j_all[0]))

lat shape: (295, 429)
lon shape: (295, 429)
points shape: (126555, 2)
i_all shape: (126555,)
j_all shape: (126555,)
Beispiel Punkt 0: [-0.817 41.183] -> Grid-Index: (np.int64(0), np.int64(0))


## 8. Beispiel: Nächstgelegene Zelle suchen

Setze `query_lon` und `query_lat` auf eine Koordinate, um die nächste Gridzelle zu finden.


In [7]:
# Beispiel: erste vorhandene Grid-Koordinate als Testpunkt verwenden
query_lon = float(lon.ravel()[0])
query_lat = float(lat.ravel()[0])

distance, nearest_flat_idx = tree.query([query_lon, query_lat])
i, j = np.unravel_index(nearest_flat_idx, lat.shape)

print("Query:", (query_lon, query_lat))
print("Nächste Zelle i/j:", (i, j))
print("Koordinate der Zelle:", (float(lon[i, j]), float(lat[i, j])))
print("Distanz im lon/lat-Raum:", distance)

Query: (-0.817, 41.183)
Nächste Zelle i/j: (np.int64(0), np.int64(0))
Koordinate der Zelle: (-0.817, 41.183)
Distanz im lon/lat-Raum: 0.0
